# Figuras del proyecto con datos y modelos reales

Notebook **didáctico**, como el anterior, pero este sí lee la capa plata y carga los modelos
entrenados. Por eso hay que ejecutarlo **después** del notebook 05.

Reúne las representaciones visuales que no forman parte del flujo de trabajo pero ayudan a
entenderlo: el árbol dibujado, los coeficientes de la regresión logística en escala legible, las
curvas de evaluación y las distribuciones que sostienen dos de las decisiones de preparación.

En el proceso principal estas vistas no son necesarias —allí basta con la métrica— pero puestas
una al lado de otra explican de un vistazo qué está haciendo el sistema.

| Figura | Qué muestra | De dónde sale |
|---|---|---|
| P1 | El árbol de decisión real, podado a tres niveles | Se reentrena aquí, con los mismos hiperparámetros |
| P2 | Los coeficientes de la logística como razones de cuotas | Se reentrena aquí |
| P3 | Curva precisión-recall con las dos líneas base | Modelo de producción sobre el conjunto de prueba |
| P4 | Matriz de confusión al umbral elegido | Modelo de producción |
| P5 | Distribuciones: por qué el salario no tiene atípicos y el saldo sí tiene estructura | Capa plata |
| P6 | El reparto del cupo por país | Capa oro |

**Aviso:** este notebook **no escribe ninguna tabla**. Solo lee y dibuja.


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix,
                             precision_recall_curve)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, plot_tree

CHURN, SAFE, ACC, GREY = "#a8471f", "#2c5a72", "#8a5a08", "#78736a"
INK, PAPER = "#1c1a17", "#efe9dd"
CAJA = dict(boxstyle="round,pad=0.28", facecolor="white", edgecolor="none", alpha=0.85)

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 160, "savefig.bbox": "tight",
    "axes.grid": True, "grid.alpha": 0.22,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
})

def caja_bigotes(ax, datos, horizontal=False, **kw):
    """Compatibilidad: en matplotlib 3.10 `vert` quedó en desuso frente a `orientation`."""
    try:
        return ax.boxplot(datos, orientation="horizontal" if horizontal else "vertical", **kw)
    except TypeError:
        return ax.boxplot(datos, vert=not horizontal, **kw)


import matplotlib
print("matplotlib:", matplotlib.__version__)

SEED = 42
CATALOG = "bank_churn"

df = spark.table(f"{CATALOG}.silver.bank_customers_clean").toPandas()
df["exited"] = df["exited"].astype(bool)
print(f"{len(df):,} clientes · tasa base {df['exited'].mean():.2%}")

In [ ]:
def preparar_destino():
    """Misma carpeta que el notebook de conceptos."""
    try:
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.gold.figuras")
        destino = f"/Volumes/{CATALOG}/gold/figuras"
        os.makedirs(destino, exist_ok=True)
    except Exception as e:
        destino = "/tmp/figuras"
        os.makedirs(destino, exist_ok=True)
        print(f"[aviso] sin volumen ({type(e).__name__}); se usa {destino}")
    print("destino:", destino)
    return destino


DESTINO = preparar_destino()


def guardar(fig, nombre):
    ruta = os.path.join(DESTINO, f"{nombre}.png")
    fig.savefig(ruta)
    print(f"  guardada: {ruta}")
    return ruta

---

## P1 · El árbol de decisión, dibujado

Un árbol de decisión tiene una propiedad que los demás modelos del proyecto no tienen: se puede
dibujar entero y leer sin saber estadística. Es la representación más directa de qué ha aprendido
el sistema.

Se reentrena con **los mismos hiperparámetros ganadores** que encontró la búsqueda en rejilla
—profundidad 8, mínimo 30 clientes por hoja, criterio de entropía— y se poda a tres niveles para
que quepa en una pantalla.

**Cómo se lee cada nodo:** la primera línea es la pregunta, y si se cumple se baja por la
izquierda. `samples` es qué porcentaje de los clientes llega ahí, `value` cómo se reparten entre
las dos clases, y `class` qué decidiría el árbol si parase en ese punto.


In [ ]:
CAT_ARBOL = ["geography", "gender"]
NUM_ARBOL = ["credit_score", "balance", "tenure", "estimated_salary", "age", "num_of_products"]
BIN       = ["balance_zero", "is_active_member", "has_cr_card"]

X = df[CAT_ARBOL + NUM_ARBOL + BIN].copy()
for c in BIN:
    X[c] = X[c].astype(int)
y = df["exited"]

prep_arbol = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), CAT_ARBOL),
    ("num", "passthrough", NUM_ARBOL),
    ("bin", "passthrough", BIN),
], remainder="drop", verbose_feature_names_out=False)

arbol = Pipeline([("prep", prep_arbol),
                  ("clf", DecisionTreeClassifier(criterion="entropy", max_depth=8,
                                                 min_samples_leaf=30,
                                                 class_weight="balanced",
                                                 random_state=SEED))]).fit(X, y)

clf = arbol.named_steps["clf"]
nombres = list(arbol.named_steps["prep"].get_feature_names_out())

fig, ax = plt.subplots(figsize=(20, 8.5))
plot_tree(clf, max_depth=3, feature_names=nombres,
          class_names=["se queda", "abandona"], filled=True, rounded=True,
          proportion=True, impurity=False, fontsize=8, ax=ax)
ax.set_title("Figura P1 · El árbol de decisión ajustado, podado a tres niveles", loc="left")
ax.grid(False)
plt.tight_layout(); guardar(fig, "P1_arbol_podado"); plt.show()

print(f"profundidad real          : {clf.get_depth()}")
print(f"hojas                     : {clf.get_n_leaves()}")
print(f"mínimo de clientes por hoja: {clf.min_samples_leaf}")
print("\nEsta es la figura que un gestor de retención puede leer entera.")

---

## P2 · Los coeficientes de la logística, como razones de cuotas

La regresión logística no gana en average precision, pero es la única del proyecto cuyos
coeficientes se pueden poner sobre una mesa y explicar uno a uno.

Un coeficiente vive en escala de logaritmo de cuotas y no le dice nada a nadie. Elevando **e** a
esa potencia se convierte en un **multiplicador**: un 2,2 duplica largamente las cuotas de
abandono, un 0,45 las deja en menos de la mitad.

**Dos avisos para leerla bien.** Las categóricas se comparan siempre contra su categoría de
referencia, que es la que eliminó `drop="first"`. Y las numéricas están escaladas, así que su
razón de cuotas es **por cada desviación típica**, no por cada euro ni por cada año.


In [ ]:
CAT_LINEAL = ["geography", "gender", "age_group", "products_group"]
NUM_LINEAL = ["credit_score", "balance", "tenure", "estimated_salary"]

XL = df[CAT_LINEAL + NUM_LINEAL + BIN].copy()
for c in BIN:
    XL[c] = XL[c].astype(int)

prep_lineal = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), CAT_LINEAL),
    ("num", StandardScaler(), NUM_LINEAL),
    ("bin", "passthrough", BIN),
], remainder="drop", verbose_feature_names_out=False)

log = Pipeline([("prep", prep_lineal),
                ("clf", LogisticRegression(C=10.0, class_weight="balanced",
                                           max_iter=2000, random_state=SEED))]).fit(XL, y)

lclf = log.named_steps["clf"]
nom_log = list(log.named_steps["prep"].get_feature_names_out())

coef = pd.DataFrame({"variable": nom_log,
                     "coeficiente": lclf.coef_[0],
                     "odds_ratio": np.exp(lclf.coef_[0])})
coef = coef.reindex(coef.odds_ratio.sub(1).abs().sort_values().index)

fig, ax = plt.subplots(figsize=(10, 0.42 * len(coef) + 2))
pos = np.arange(len(coef))
colores = [CHURN if v > 1 else SAFE for v in coef.odds_ratio]
ax.barh(pos, coef.odds_ratio - 1, left=1, color=colores, height=0.62, zorder=2)
ax.axvline(1, color=INK, lw=1.6)

for i, (v, n) in enumerate(zip(coef.odds_ratio, coef.variable)):
    lado = 1.02 if v > 1 else 0.98
    ax.text(v + (0.06 if v > 1 else -0.06), i, f"{v:.2f}",
            va="center", ha="left" if v > 1 else "right",
            fontsize=8.5, fontweight="bold", color=INK)

ax.set_yticks(pos); ax.set_yticklabels(coef.variable, fontsize=9)
ax.set_xlabel("razón de cuotas   (1 = la variable no cambia nada)")
ax.set_title("Figura P2 · Coeficientes de la regresión logística, en escala legible", loc="left")
ax.text(0.99, 0.02, "a la izquierda: reduce el riesgo", transform=ax.transAxes,
        ha="right", fontsize=8.5, color=SAFE, bbox=CAJA)
plt.tight_layout(); guardar(fig, "P2_odds_ratios"); plt.show()

print(coef.round(4).to_string(index=False))
print(f"\nintercepto: {lclf.intercept_[0]:.4f}")

---

## P3 · La curva precisión-recall, medida donde toca

El suelo del azar es la tasa base, no cero. Esta figura lo hace visible y sitúa a la regla de
negocio sobre la misma curva.

**Aviso metodológico, y es el importante.** Esta figura se calcula sobre el **conjunto de prueba**,
las 2.000 filas que el modelo nunca vio. Medirla sobre la cartera completa daría una average
precision mucho más alta —alrededor de 0,82 frente a 0,70— porque el modelo ya había visto el 80 %
de esas filas al entrenarse. Ese número sería real y sería inútil: no estima lo que pasará con
clientes nuevos, sino lo bien que el modelo recuerda los que ya conoce.

**La comprobación que lo delata:** la regla de negocio da prácticamente lo mismo dentro y fuera de
la muestra, porque es una tabla de diez tasas y no tiene nada que memorizar. Si un número se mueve
al cambiar de conjunto y el otro no, el que se mueve es el que estaba inflado.


In [ ]:
# El conjunto de prueba, tal y como lo aparto el notebook 04
prueba = spark.table(f"{CATALOG}.silver.test_holdout").toPandas()
prueba["exited"] = prueba["exited"].astype(bool)
y_prueba = prueba["exited"]
print(f"conjunto de prueba: {len(prueba):,} filas · {int(y_prueba.sum())} abandonos")

# Regla de negocio: tasa historica por segmento edad x actividad.
# Las tasas se aprenden SOLO sobre entrenamiento, igual que en el notebook 04.
entrena = df[~df["customer_id"].isin(prueba["customer_id"])]
clave_e = entrena["age_group"].astype(str) + "|" + entrena["is_active_member"].astype(int).astype(str)
tasas   = entrena.groupby(clave_e)["exited"].mean()
clave_p = prueba["age_group"].astype(str) + "|" + prueba["is_active_member"].astype(int).astype(str)
p_regla = clave_p.map(tasas).fillna(entrena["exited"].mean()).to_numpy()

# El modelo de produccion, cargado del registro. Si no esta, se reentrena SOLO con entrenamiento.
try:
    import mlflow
    mlflow.set_registry_uri("databricks-uc")
    ref = spark.table(f"{CATALOG}.gold.modelo_produccion").toPandas().iloc[0]
    modelo = mlflow.sklearn.load_model(f"models:/{ref['uc_model']}/{ref['uc_version']}")
    p_modelo = modelo.predict_proba(prueba)[:, 1]
    origen = "modelo de produccion cargado de Unity Catalog"
except Exception as e:
    from sklearn.ensemble import RandomForestClassifier
    Xe = entrena[CAT_ARBOL + NUM_ARBOL + BIN].copy()
    for c in BIN:
        Xe[c] = Xe[c].astype(int)
    Xp = prueba[CAT_ARBOL + NUM_ARBOL + BIN].copy()
    for c in BIN:
        Xp[c] = Xp[c].astype(int)
    bosque = Pipeline([("prep", prep_arbol),
                       ("clf", RandomForestClassifier(n_estimators=600, max_depth=10,
                                                      min_samples_leaf=5, max_features=0.7,
                                                      class_weight="balanced_subsample",
                                                      random_state=SEED, n_jobs=-1))]).fit(Xe, entrena["exited"])
    p_modelo = bosque.predict_proba(Xp)[:, 1]
    origen = f"bosque reentrenado solo con entrenamiento ({type(e).__name__} al cargar el registrado)"

print("origen de las probabilidades:", origen)

prec, rec, _ = precision_recall_curve(y_prueba, p_modelo)
ap = average_precision_score(y_prueba, p_modelo)
ap_regla = average_precision_score(y_prueba, p_regla)
tasa_base = y_prueba.mean()

fig, ax = plt.subplots(figsize=(9.5, 5.4))
ax.plot(rec, prec, color=CHURN, lw=2.8, zorder=3, label=f"modelo   ·   AP = {ap:.4f}")
ax.fill_between(rec, tasa_base, prec, color=CHURN, alpha=0.10, zorder=1)

pr_r, rc_r, _ = precision_recall_curve(y_prueba, p_regla)
ax.plot(rc_r, pr_r, color=ACC, lw=2.2, zorder=3,
        label=f"regla de negocio   ·   AP = {ap_regla:.4f}")

ax.axhline(tasa_base, color=GREY, ls="--", lw=1.8, zorder=2)
ax.text(0.02, tasa_base - 0.045, f"suelo del azar = tasa base = {tasa_base:.4f}",
        fontsize=9, color=INK, bbox=CAJA, zorder=6)

ax.set_xlabel("recall (cobertura)"); ax.set_ylabel("precisión")
ax.set_xlim(0, 1); ax.set_ylim(0, 1.02)
ax.set_title("Figura P3 · Curva precisión-recall sobre el conjunto de prueba", loc="left")
ax.legend(fontsize=9.5, loc="upper right")
plt.tight_layout(); guardar(fig, "P3_curva_pr"); plt.show()

print(f"\nAP del modelo sobre PRUEBA          : {ap:.4f}")
print(f"AP de la regla sobre PRUEBA         : {ap_regla:.4f}")
print(f"margen                              : {ap - ap_regla:+.4f}")

---

## P4 · La matriz de confusión al umbral por capacidad

Con capacidad fija, el umbral no se elige por coste: se ordena a todos los clientes y se toman los
primeros hasta agotar el cupo. Sobre el conjunto de prueba, que es la quinta parte de la cartera,
el cupo proporcional son 160 contactos.


In [ ]:
CAPACIDAD_TEST = 160          # 800 contactos sobre la cartera completa, a escala del test
umbral = np.sort(p_modelo)[-CAPACIDAD_TEST]
pred = p_modelo >= umbral

cm = confusion_matrix(y_prueba, pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7.2, 5.4))
ax.imshow(cm, cmap="OrRd", aspect="auto")
etiquetas = [["verdaderos negativos", "falsos positivos"],
             ["falsos negativos", "verdaderos positivos"]]
for i in range(2):
    for j in range(2):
        ax.text(j, i - 0.10, f"{cm[i, j]:,}".replace(",", "."), ha="center", va="center",
                fontsize=19, fontweight="bold",
                color="white" if cm[i, j] > cm.max() * 0.5 else INK)
        ax.text(j, i + 0.20, etiquetas[i][j], ha="center", va="center", fontsize=8.5,
                color="white" if cm[i, j] > cm.max() * 0.5 else GREY)

ax.set_xticks([0, 1]); ax.set_xticklabels(["no se contacta", "se contacta"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["se queda", "abandona"])
ax.set_xlabel("decisión del sistema"); ax.set_ylabel("lo que ocurrió de verdad")
ax.set_title(f"Figura P4 · Matriz de confusión con {CAPACIDAD_TEST} contactos sobre prueba", loc="left")
ax.grid(False)
plt.tight_layout(); guardar(fig, "P4_matriz_confusion"); plt.show()

techo = CAPACIDAD_TEST / int(y_prueba.sum())
print(f"umbral efectivo por capacidad : {umbral:.4f}")
print(f"abandonos captados            : {tp} de {int(y_prueba.sum())}")
print(f"precisión de la campaña       : {tp / pred.sum():.1%}")
print(f"techo de recall alcanzable    : {techo:.4f}")
print(f"porcentaje del techo          : {(tp / int(y_prueba.sum())) / techo:.1%}")
print(f"\nExtrapolado a 800 contactos sobre la cartera: {tp * 5} abandonos, {(tp*145 - fp*35)*5:,.0f} EUR")

---

## P5 · Distribuciones: la figura que faltaba en el análisis exploratorio

El análisis exploratorio del proyecto es deliberadamente bivariante: los hallazgos que importan
salen de cruzar variables, no de mirarlas por separado. Aun así, hay dos distribuciones que ganan
mucho al verse, porque son la prueba visual de dos decisiones de preparación de datos.

**Arriba:** el salario estimado es plano como una mesa. Es un sorteo uniforme, no un salario.

**Abajo:** el saldo tiene una masa puntual en cero, **un hueco vacío hasta 3.769**, y después una
campana. Esa es la prueba de que el cero es un estado y no un dato faltante — y hasta ahora solo
existía como un número dentro de un `describe()`.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 7),
                         gridspec_kw={"width_ratios": [2.4, 1]})

# --- salario ---
axes[0, 0].hist(df["estimated_salary"], bins=60, color=CHURN, alpha=0.8,
                edgecolor="white", lw=0.4)
axes[0, 0].set_title("Salario estimado: plano como una mesa", loc="left", fontsize=11.5)
axes[0, 0].set_xlabel("salario estimado"); axes[0, 0].set_ylabel("clientes")
axes[0, 0].text(0.5, 0.86, "una uniforme perfecta.\nNingún salario real se reparte así",
                transform=axes[0, 0].transAxes, ha="center", fontsize=9.5,
                color=INK, bbox=CAJA)

bp = caja_bigotes(axes[0, 1], df["estimated_salary"], widths=0.5, patch_artist=True,
                  flierprops=dict(marker="o", ms=3, mfc=CHURN, mec="none"))
bp["boxes"][0].set(facecolor=CHURN, alpha=0.35)
for k in ("whiskers", "caps", "medians"):
    for e in bp[k]: e.set(color=INK, lw=1.3)
n_out = len(bp["fliers"][0].get_ydata())
axes[0, 1].set_xticks([])
axes[0, 1].set_title(f"{n_out} atípicos", loc="left", fontsize=11.5)
axes[0, 1].grid(False)
axes[0, 1].text(0.5, 0.06, "cero, y está garantizado\npor construcción",
                transform=axes[0, 1].transAxes, ha="center", fontsize=8.5,
                color=INK, bbox=CAJA)

# --- saldo ---
axes[1, 0].hist(df["balance"], bins=70, color=SAFE, alpha=0.85, edgecolor="white", lw=0.4)
axes[1, 0].set_title("Saldo: dos poblaciones, no una con faltantes", loc="left", fontsize=11.5)
axes[1, 0].set_xlabel("saldo"); axes[1, 0].set_ylabel("clientes")
min_pos = df.loc[df.balance > 0, "balance"].min()
axes[1, 0].axvspan(0, min_pos, color=CHURN, alpha=0.18)
axes[1, 0].text(min_pos * 1.15, axes[1, 0].get_ylim()[1] * 0.55,
                f"hueco vacío\nde 0 a {min_pos:,.0f}".replace(",", "."),
                fontsize=9.5, color=INK, bbox=CAJA)

cero = (df.balance == 0).mean() * 100
axes[1, 1].bar([0, 1], [cero, 100 - cero], color=[CHURN, SAFE], width=0.6)
axes[1, 1].set_xticks([0, 1]); axes[1, 1].set_xticklabels(["saldo = 0", "saldo > 0"])
axes[1, 1].set_ylabel("% de clientes")
axes[1, 1].set_title("El cero no es raro: es un tercio", loc="left", fontsize=11.5)
for i, v in enumerate([cero, 100 - cero]):
    axes[1, 1].text(i, v + 1.5, f"{v:.1f}%", ha="center", fontweight="bold", fontsize=10)

fig.suptitle("Figura P5 · Las dos distribuciones que sostienen dos conclusiones del proyecto",
             fontsize=12.5, fontweight="bold", x=0.008, ha="left")
plt.tight_layout(); guardar(fig, "P5_distribuciones"); plt.show()

print(f"saldo positivo más bajo de toda la base : {min_pos:,.2f}")
print(f"clientes con saldo exactamente cero     : {(df.balance == 0).sum():,} ({cero:.1f} %)")
print(f"mínimo del salario estimado             : {df.estimated_salary.min():,.2f}")

---

## P6 · El reparto del cupo por país

La decisión de equidad convertida en números. Como la tasa base difiere entre países, un umbral
global haría que Alemania copara las plazas. Repartir el cupo en proporción a los abandonos
esperados produce **umbrales efectivos distintos por país**.

**Lo que hay que leer:** un alemán necesita puntuar más alto que un español para entrar en la
campaña. No hay respuesta técnica correcta a esto: se cuantifica lo que cuesta y la decisión se
devuelve al negocio.

*A diferencia de P3 y P4, esta figura sí se calcula sobre la cartera completa, porque es lo que
hace el sistema en producción: puntúa a los diez mil y reparte el cupo entre ellos.*


In [ ]:
# Aqui si se puntua la cartera completa: es lo que hace el notebook 06 en produccion
try:
    p_cartera = modelo.predict_proba(df)[:, 1]
except NameError:
    p_cartera = bosque.predict_proba(X)[:, 1]
prob = pd.Series(p_cartera, index=df.index)
esperados = prob.groupby(df["geography"]).sum()
cuota = (esperados / esperados.sum() * 800).round().astype(int)

filas = []
for pais, k in cuota.items():
    m = df["geography"] == pais
    sub = prob[m].sort_values(ascending=False)
    umbral_pais = sub.iloc[k - 1] if k > 0 and k <= len(sub) else np.nan
    sel = sub.head(k).index
    filas.append({"pais": pais, "clientes": int(m.sum()),
                  "abandonos_esperados": round(esperados[pais]),
                  "cupo": k, "umbral_efectivo": umbral_pais,
                  "captados": int(df.loc[sel, "exited"].sum())})
rep = pd.DataFrame(filas).set_index("pais").sort_values("umbral_efectivo", ascending=False)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12.5, 4.6))
x = np.arange(len(rep))

ax1.bar(x, rep["umbral_efectivo"], color=[CHURN, ACC, SAFE][:len(rep)], width=0.55, zorder=2)
for i, v in enumerate(rep["umbral_efectivo"]):
    ax1.text(i, v + 0.015, f"{v:.4f}", ha="center", fontweight="bold", fontsize=10, color=INK)
ax1.set_xticks(x); ax1.set_xticklabels(rep.index)
ax1.set_ylabel("probabilidad mínima para entrar en la campaña")
ax1.set_ylim(0, max(rep["umbral_efectivo"]) * 1.25)
ax1.set_title("Umbral efectivo por país", loc="left", fontsize=11.5)

ax2.bar(x, rep["cupo"], color=GREY, width=0.55, label="cupo asignado", zorder=2)
ax2.bar(x, rep["captados"], color=CHURN, width=0.34, label="abandonos captados", zorder=3)
for i, (c, k) in enumerate(zip(rep["captados"], rep["cupo"])):
    ax2.text(i, k + 6, f"{k}", ha="center", fontsize=9, color=GREY)
    ax2.text(i, c / 2, f"{c}", ha="center", fontsize=9.5, color="white", fontweight="bold")
ax2.set_xticks(x); ax2.set_xticklabels(rep.index)
ax2.set_ylabel("clientes"); ax2.legend(fontsize=9)
ax2.set_title("Reparto del cupo y resultado", loc="left", fontsize=11.5)

fig.suptitle("Figura P6 · La decisión de equidad, convertida en números",
             fontsize=12.5, fontweight="bold", x=0.008, ha="left")
plt.tight_layout(); guardar(fig, "P6_reparto_por_pais"); plt.show()

print(rep.round(4).to_string())

---

## Resumen

Seis figuras con datos y modelos reales, guardadas junto a las conceptuales en el volumen
`bank_churn.gold.figuras`.

Las dos primeras son las que más rinden al explicar el sistema a alguien de negocio: el árbol se
lee entero sin saber estadística, y los coeficientes en razones de cuotas convierten el modelo
lineal en frases que se pueden decir en voz alta.

Y las distribuciones de P5 son la prueba visual del hueco entre 0 y 3.769 en el saldo, que es lo
que distingue un estado de un dato faltante.
